# 实验四 · 激活函数 Sigmoid —— 高阶 API 与精度取舍

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：50–60 分钟

前三个实验的计算部分都极为简单：一条 `Add`、一条 `ReduceSum` 即可完成。本实验转向**复合函数**。Sigmoid 的数学表达式只有一行，落到基础 API 上却需要四条指令串联，并需要额外的中间缓冲。由此引出两个新问题：高阶 API 如何把这条链收敛为一次调用，以及当中间量超出数据类型的表示范围时会发生什么。

> **实验说明**
> 1. 本实验的核心内容有三点：复合函数的基础 API 组合与中间缓冲、高阶 API 的用法与临时空间约定、以及 `half` 带来的吞吐收益与两类数值失效。
> 2. 四个版本构成**两条线索**而非一条：v1 → v2 考察编程效率，v1 → v3 → v4 考察精度与吞吐的取舍。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 四个版本的核函数、CPU 基准、数据生成、精度校验与 `main` 都写在同一个 `.asc` 文件中，由一条 `bisheng` 命令编译为单个可执行程序。
> 6. 元素总数与输入取值范围都是**运行时参数**，参数扫描不需要重新编译。
> 7. 本实验的**精度判定分为两层**：`PASS` / `FAIL` 依据**绝对误差**判定，容差按「链上精度最低的那一条接口」选取；**相对误差**单独统计并按输入区间分段作图，不参与判定。
> 8. 输入取值范围默认为 [−20, 20]，目的是覆盖 `half` 会发生中间量溢出的区间。
> 9. 本实验建立在**实验三 ReduceSum** 的基础之上，其中 `TBuf` 临时缓冲的用法直接沿用。

## 🎯 学习目标

完成本实验后，开发者应能够：

- 用基础矢量 API 组合实现复合函数，并正确管理其所需的中间缓冲
- 说明为何复合计算需要 `VECCALC` 位置的临时空间，以及如何核算其大小
- 掌握高阶 API `Sigmoid` 的用法，说明其相对于手工组合的取舍
- 理解「接口框架申请临时空间」的机制，说明为何不能把片上缓冲用尽
- 掌握 `Cast` 接口与混合精度的实现方法
- 定量说明 `half` 相对 `float` 在**搬运量**与**表示范围**两方面的差异
- 识别**中间量溢出**这一类缺陷：绝对误差可能很小，相对误差却极大
- 区分「可修复的算法缺陷」与「数据格式的固有极限」，并说明区分二者的判据
- 依据实现链上精度最低的一环选择容差，说明绝对误差与相对误差各自的适用场景

## 🗺️ 学习路径

1. **准备阶段**：把 Sigmoid 分解为四条基础 API，理解中间缓冲的来源与中间量的量程
2. **知识铺垫**：高阶 API 的两种临时空间取得方式；`half` 的一项收益与两项代价
3. **Device 侧实现**：三个算子类覆盖四个版本
4. **Host 侧实现**：两套输入各配一份参考真值，双口径校验与分段误差统计
5. **编译运行**：一条 `bisheng` 命令，一次运行产出全部对照数据
6. **结果可视化**：以 CPU 与 v1 两套基线分别计算加速比
7. **误差分析**：绘制相对误差随 x 的分布，定位 v3 的失效区间
8. **参数扫描**：核数扫描对照 float 与 half 两个版本
9. **结果分析**：把搬运量、片上占用与数值正确性三件事收束到同一处讨论

## 1. 背景与动机：复合函数的实现

Sigmoid 是神经网络中最基础的激活函数之一：

$$ y = \sigma(x) = \frac{1}{1 + e^{-x}} $$

在 CPU 上，这一行公式对应一行 C 代码。在 AI Core 上，它必须被分解为四条矢量指令：

<img src="images/06.04_sigmoid_chain.png" alt="Sigmoid 的四条基础 API 组合链" width="1000">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | 基础 API | 含义 |
| --- | --- | --- |
| 1 | `Muls(t, x, -1)` | t = −x |
| 2 | `Exp(t, t)` | t = e^(−x) |
| 3 | `Adds(t, t, 1)` | t = 1 + e^(−x) |
| 4 | `Reciprocal(y, t)` | y = 1 / t |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">步骤</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">基础 API</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Muls(t, x, -1)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">t = −x</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Exp(t, t)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">t = e^(−x)</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Adds(t, t, 1)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">t = 1 + e^(−x)</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Reciprocal(y, t)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">y = 1 / t</td>
</tr>
</tbody>
</table>

由此产生两项新的工程负担。

**其一，需要中间缓冲。** 上式中的 t 既不是输入也不是输出，它只在 `Compute` 内部存在，因此应当分配为 `VECCALC` 位置的 `TBuf`——判据与实验三中 `ReduceSum` 的临时空间完全相同：**数据是否需要跨流水任务传递**。

第二步的 `Exp` 与其余三条并不同量级。官方在介绍矢量计算 API 时以它为例说明按 repeat 与 datablock 迭代的过程：

<img src="images/06.04_exp_datablock.png" alt="06.04_exp_datablock" width="820px">

*单次迭代内的 8 个 datablock 进行 Exp 计算示意图*

官方另有一条与本实验直接相关的性能说明：**`Exp`、`Ln` 接口处理同样数量的 `half` 与 `float` 数据，耗时是一样的**，其内部对 `float` 做了优化。§13 ② 会用到这一条。

**其二，中间量的取值范围可能远大于输入与输出。** 输入 x ∈ [−20, 20]，输出 y ∈ (0, 1)，二者都在很小的范围内；而中间量 e^(−x) 的取值范围是 [e^(−20), e^(20)] ≈ [2×10^-9, 4.9×10^8]，跨越了 17 个数量级。

**数据类型能否容纳中间量，与它能否容纳输入输出，是两个独立的问题。** 这正是 v3 与 v4 的主题。

### 1.1 算法与数据规格

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 项目 | 取值 | 说明 |
| --- | --- | --- |
| 元素总数 N | 2^21 = 2 097 152 | **运行时参数**，可由命令行覆盖 |
| 输入取值范围 | [−20, 20] 上的均匀分布 | **运行时参数**，取到 ±20 是为了覆盖 `half` 的失效区间 |
| 输出 | (0, 1) | 单调、有界 |
| 分块长度 TILE_LENGTH | 4096 个元素 | 编译期常量，可由 `-D` 覆盖 |
| 参与计算的核数 | 8 | 编译期常量，可由 `-D` 覆盖 |
| v1 / v2 的数据类型 | `float` | 每元素搬运 8 字节（读 4 写 4） |
| v3 / v4 的数据类型 | `half` | 每元素搬运 **4 字节** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">项目</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">取值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">元素总数 N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2^21 = 2 097 152</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>运行时参数</strong>，可由命令行覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入取值范围</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">[−20, 20] 上的均匀分布</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>运行时参数</strong>，取到 ±20 是为了覆盖 <code>half</code> 的失效区间</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">(0, 1)</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单调、有界</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分块长度 TILE_LENGTH</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4096 个元素</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量，可由 <code>-D</code> 覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参与计算的核数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量，可由 <code>-D</code> 覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 / v2 的数据类型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>float</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每元素搬运 8 字节（读 4 写 4）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 / v4 的数据类型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>half</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每元素搬运 <strong>4 字节</strong></td>
</tr>
</tbody>
</table>

## 2. 高阶 API 与临时空间

Ascend C 的类库 API 是分层组织的。官方的编程接口概览图自下而上给出四层：

<img src="images/06.04_api_layers.png" alt="06.04_api_layers" width="900px">

*Ascend C 编程类库 API 示意图*

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 层次 | 抽象粒度 | 本实验涉及的内容 |
| --- | --- | --- |
| 语言扩展层 | — | 未涉及 |
| 基础 API（SIMD） | **单指令** | `Muls`、`Exp`、`Adds`、`Reciprocal`、`Cast` |
| 高阶 API | **单核公共算法** | **激活函数**一类中的 `Sigmoid` |
| 算子模板库 | 多核算子样例 | 未涉及 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">层次</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">抽象粒度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验涉及的内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">语言扩展层</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未涉及</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础 API（SIMD）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>单指令</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Muls</code>、<code>Exp</code>、<code>Adds</code>、<code>Reciprocal</code>、<code>Cast</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>单核公共算法</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>激活函数</strong>一类中的 <code>Sigmoid</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子模板库</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多核算子样例</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未涉及</td>
</tr>
</tbody>
</table>

官方文档对这两层的定位是：基础 API「实现对硬件能力的抽象，开放芯片的能力」，高阶 API 则是「基于单核公共算法抽象」的封装；并明确指出**高阶 API 通过调用基础 API 来实现功能**。

上图中高阶 API 一行列出了「数学计算 / 矩阵计算 / **激活函数** / 池化计算 / 索引计算 / 通信编程」等类别，`Sigmoid` 属于其中的激活函数一类——**本实验用高阶 API 实现 Sigmoid，走的正是官方给出的路线**：

```cpp
AscendC::Sigmoid(yLocal, xLocal, TILE_LENGTH);   // 替代上面四条指令
```

### 2.1 高阶 API 同样需要临时空间

高阶 API 的内部实现涉及复合计算，同样需要中间变量。它提供两种临时空间的取得方式：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 方式 | 写法 | 特点 |
| --- | --- | --- |
| 开发者传入 | `Sigmoid(dst, src, sharedTmpBuffer, count)` | 空间由开发者管理，可复用，利用率高 |
| 接口框架申请 | `Sigmoid(dst, src, count)` | 无须申请，但**必须预留**未分配的片上空间 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">方式</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">写法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">特点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">开发者传入</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Sigmoid(dst, src, sharedTmpBuffer, count)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">空间由开发者管理，可复用，利用率高</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">接口框架申请</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Sigmoid(dst, src, count)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无须申请，但<strong>必须预留</strong>未分配的片上空间</td>
</tr>
</tbody>
</table>

本实验的 v2 采用第二种写法，因为它不引入需要人工估算的数值。但这带来一条必须遵守的约束：**不能把片上缓冲分配殆尽**。若 `TPipe` 已把 Unified Buffer 全部分给各个队列与 `TBuf`，框架将无空间可用。

v2 的显式占用为输入队列 32 KB 加输出队列 32 KB，合计 64 KB，占 UB（192 KB）的 33%，余量充足。若在实际工程中把片上缓冲分配殆尽，则须改用传入形式，并在 Host 侧先取得所需空间的范围。

> **这一取得方式是官方的通用约定。** 官方为需要临时空间的高阶 API 统一提供了一组 `Getxxx MaxMinTmpSize` 形式的 Host 侧接口，用于查询该接口在给定 shape 与数据类型下所需临时空间的最小值与最大值；另有 `GetxxxTmpBufferFactorSize` 给出「同时存活的中间节点数」与「额外开销」两个系数，供开发者反推单次可处理的数据量。`GetSigmoidMaxMinTmpSize` 即这一命名规则在 `Sigmoid` 上的实例；其具体原型与参数说明见《Ascend C API 参考》，不在本课程语料范围内。

### 2.2 手工组合与高阶 API 的取舍

官方以矩阵乘为例给出了两者的封装关系：高阶 API 把 CopyIn、切分、Compute、聚合、CopyOut 这一整套流程收在内部，开发者只面对一次调用。Sigmoid 与之同构，只是内部封装的是四条矢量指令而非一套矩阵乘流程。

<img src="images/06.04_base_vs_high_api.png" alt="06.04_base_vs_high_api" width="600px">

*基础 API 与高阶 API 的对照图*

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 对比项 | 手工组合基础 API | 高阶 API |
| --- | --- | --- |
| 代码量 | 四条指令 + 一块中间缓冲 | 一次调用 |
| 可控性 | 完全可控，可针对性优化 | 由实现决定 |
| 跨硬件兼容性 | 基础 API 中标注 **ISASI**（硬件体系结构相关的接口）的部分不保证跨硬件版本兼容 | 保证兼容 |
| 数值处理 | 由开发者负责 | 由实现负责 |
| 地址重叠 | 基础算术接口未禁止源与目的重叠 | **明确禁止**源、目的与临时空间互相重叠 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对比项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">手工组合基础 API</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">高阶 API</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">代码量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四条指令 + 一块中间缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一次调用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可控性</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">完全可控，可针对性优化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由实现决定</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">跨硬件兼容性</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础 API 中标注 <strong>ISASI</strong>（硬件体系结构相关的接口）的部分不保证跨硬件版本兼容</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">保证兼容</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数值处理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由开发者负责</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由实现负责</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">地址重叠</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础算术接口未禁止源与目的重叠</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>明确禁止</strong>源、目的与临时空间互相重叠</td>
</tr>
</tbody>
</table>

第三行中的 ISASI 是 Instruction Set Architecture Special Interface 的缩写，官方的原话是：基础 API 中「标注为 ISASI 类别的 API，不能保证跨硬件版本兼容」。

倒数第二行值得注意：手工组合意味着**溢出、下溢、边界情形，以及链上每一条接口自身的实现精度，都由开发者自己承担**。v3 会给出溢出的例子；而在精度上，本实验中手工组合与高阶 API 的精度相差若干个数量级，差别来自链上的一条低精度接口。

最后一行的差异源于后者内部要维护自己的中间状态。本实验各版本的 `xLocal` 来自输入队列、`yLocal` 来自输出队列，天然不重叠，因此无须额外处理；但若打算把中间结果就地写在输出缓冲上以省掉 `TBuf`，就必须先查清所用接口的规定。

## 3. half：一项收益与两项代价

`half`（IEEE 754 半精度）以 16 位表示一个浮点数，它与 `float` 的差别全部来自位域的分配：

<img src="images/06.04_half_vs_float.png" alt="float32 与 half 的位域结构对照" width="800">

上图的位宽划分出自官方的标量数据类型表：

<img src="images/06.04_scalar_dtypes.png" alt="06.04_scalar_dtypes" width="600px">

*标量数据类型。`half` 为 1 / 5 / 10、`float` 为 1 / 8 / 23、`bfloat16_t` 为 1 / 8 / 7——最后一项是动手练习第 8 题的依据。该表「取值范围」一列的上下标在文档中排版异常，请以位宽列为准*

**收益：搬运量减半。** 本算子每处理一个元素需读入 1 个、写出 1 个，改用 `half` 后每元素的搬运量由 8 字节降为 4 字节，片上占用也随之减半。**这项收益究竟能兑现多少，取决于耗时中有多大比例真正与字节数成正比**。

**代价一：输出分辨率降低。** `half` 在 1.0 附近相邻两个可表示数的间隔（1 ulp）为 2^-10 ≈ 9.8×10^-4，因此把结果舍入到 `half` 所引入的绝对误差上界为半个间隔，即 **≈ 4.9×10^-4**；输出的绝对误差不可能优于这一量级。这是数据类型本身决定的，无法通过改写算法消除。后文凡提到「输出分辨率」，指的都是这个 4.9×10^-4 的舍入误差上界。

**代价二：中间量溢出。** `half` 能表示的最大值为 65504。当 x < −11.09 时中间量 e^(−x) 超出这一上限，此后的 1/(1 + t) 便不再是正确的 Sigmoid 值。该阈值由 −ln(65504) 得出。

> 超出表示范围之后得到的是饱和值还是 inf，由计算单元决定。官方的说明是：输入 inf/nan 或计算结果超出范围时，**AI Core 仅支持饱和模式，Vector Core 仅支持 inf/nan 模式**。两种模式下 1/(1 + t) 的结果都会偏离真值，因此下文的判据对两者同时成立——思考题第 7 题要求说明这一点。

<img src="images/06.04_three_regions.png" alt="half 的两条阈值把输入区间切成三段" width="800">

**两项代价的性质并不相同。** 代价一是数据格式的固有极限，任何实现都无法回避；代价二只是中间步骤选错了数据类型，**可以修复**——输入输出保持 `half` 以保留搬运量减半的收益，中间计算通过 `Cast` 转到 `float` 域完成。这就是 v4 的做法。

### 3.1 区分两类失效的判据

`half` 还有一条下界：其最小非规格化数约为 5.96×10^-8，当 σ(x) 小于该值时（对应 x < −16.64）结果只能是 0，与中间计算的精度无关。这一条与代价一同属格式的固有极限。

两条阈值把输入区间切成三段：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 区间 | 中间量 e^(−x) | 真值能否用 `half` 表示 | v3（纯 half） | v4（混合精度） |
| --- | --- | --- | --- | --- |
| ① x < −16.64 | 溢出 | **不能**，小于最小非规格化数 | 输出 0，相对误差为 1 | 输出 0，相对误差为 1 |
| ② −16.64 < x < −11.09 | **溢出** | 能，但只能用非规格化数 | **相对误差无上界** | **相对误差不超过 1/2** |
| ③ x > −11.09 | 不溢出 | 能，且为规格化数 | 正确 | 正确 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">区间</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">中间量 e^(−x)</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">真值能否用 <code>half</code> 表示</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v3（纯 half）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v4（混合精度）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">① x &lt; −16.64</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">溢出</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不能</strong>，小于最小非规格化数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出 0，相对误差为 1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出 0，相对误差为 1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">② −16.64 &lt; x &lt; −11.09</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>溢出</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">能，但只能用非规格化数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>相对误差无上界</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>相对误差不超过 1/2</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">③ x &gt; −11.09</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不溢出</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">能，且为规格化数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">正确</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">正确</td>
</tr>
</tbody>
</table>

区间 ① 与区间 ② 都表现为绝对误差极小、相对误差极大，在绝对误差判定下都不会被发现；但其中只有区间 ② 属于可修复的算法缺陷。**区分二者的办法是把误差按输入区间分段统计，并对区间 ② 施加一条上界判据。**

该判据为：在区间 ② 内真值不小于 `half` 非规格化数的步长，因此纯量化所致的相对误差不可能超过 **1/2**。相对误差一旦超过 1/2，就说明它来自计算过程而非输出量化。

## 4. 本实验的测量方法

性能测量沿用实验二 §5 的做法：两个指标（`cpu_ms`、`kernel_ms`）与四条要点（预热、多次重复取平均、先同步再停止计时、基准与被测使用相同的优化级别），此处不再复述。与实验二、实验三一致，本实验也不单独测量「主机拷贝 + 核函数 + 设备回传」这一端到端口径，理由见实验二 §5.1：真实计算图中数据一次搬入设备内存后会常驻其上，为单个算子各安排一次完整往返的调用方式在工程上并不存在。本节只说明本实验特有的三点。

### 4.1 两套输入各配一份参考真值

v1、v2 的输入是 `float`；v3、v4 的输入是同一批数据**量化到 `half` 之后**的值，二者并不相同。若统一用 `float` 输入算出的真值去校验 `half` 版本，就会把**输入量化的误差**记到算子头上。

因此本实验准备两份真值：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 输入 | 参考真值 |
| --- | --- | --- |
| v1 / v2 | x（float32） | σ(x)，Host 侧以 `double` 计算 |
| v3 / v4 | half(x) 解码回 float 后的值 | σ(half(x))，Host 侧以 `double` 计算 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">输入</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参考真值</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 / v2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">x（float32）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">σ(x)，Host 侧以 <code>double</code> 计算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 / v4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">half(x) 解码回 float 后的值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">σ(half(x))，Host 侧以 <code>double</code> 计算</td>
</tr>
</tbody>
</table>

这样得到的误差才是**算子本身**引入的。Host 侧的 `F32ToF16` / `F16ToF32` 两个函数用整数位运算实现，不依赖任何库，`half` 的编码规则因此也一并显式可见。

### 4.2 判定口径与分析口径分开

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 用途 | 口径 | 容差 |
| --- | --- | --- |
| `PASS` / `FAIL` 判定 | **绝对误差** | 见下表，按版本取值 |
| 误差分析 | **相对误差**，按输入区间分段统计 | 不参与判定 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">用途</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">口径</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">容差</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>PASS</code> / <code>FAIL</code> 判定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>绝对误差</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">见下表，按版本取值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">误差分析</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>相对误差</strong>，按输入区间分段统计</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不参与判定</td>
</tr>
</tbody>
</table>

容差的选取遵循一条原则：**由本实现链上精度最低的那一环决定**，而不是由数据类型的机器精度决定。按这条原则，四个版本分成两档：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 链上精度最低的一环 | 容差 | 说明 |
| --- | --- | --- | --- |
| v2 | 高阶 `Sigmoid` 的内部实现 | **1e-6** | 数值处理由实现负责，应当达到 float32 的水平 |
| v1 / v3 / v4 | 基础接口 `Reciprocal` | **5e-3** | 硬件提供的近似求倒数接口，**实测**相对误差在 10^-3 量级 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">链上精度最低的一环</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">容差</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 <code>Sigmoid</code> 的内部实现</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>1e-6</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数值处理由实现负责，应当达到 float32 的水平</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 / v3 / v4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础接口 <code>Reciprocal</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>5e-3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">硬件提供的近似求倒数接口，<strong>实测</strong>相对误差在 10^-3 量级</td>
</tr>
</tbody>
</table>

第二档需要说明：`Reciprocal` 的近似精度比 `half` 的输出舍入误差上界（4.9×10^-4）还要粗一个数量级，因此 v3、v4 的容差同样由它决定，与数据类型无关。

> **这里的 10^-3 量级是实测值，不是官方给出的规格。** 官方文档没有为基础接口给出误差界或 ULP 指标，因此这一档容差只能由本实验自己测出来（§13 ⑥ 给出定位方法）。可以参照的是官方在算子精度验收脚本中采用的门限：`loss = 1e-3`，注释为「一般 fp16 要求绝对误差和相对误差均不超过千分之一」。本实验取 5e-3，是在这一量级上留了余量。

### 4.3 CPU 基准的角色

CPU 基准是单线程 `float32` 的 `1.0f / (1.0f + expf(-x))`。它有两个用途：作为四个版本共同的性能基线；以及作为精度的一条**参照线**——它自己相对 `double` 真值也有误差，程序会把这一项一并报告。凡是 `float` 输出的版本，其误差都应当与这条参照线同量级；若某个版本高出若干个数量级，说明它的实现链上有一环拉低了整体精度。

## 5. 环境准备与检查

In [ ]:
!mkdir -p src_sigmoid

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

In [ ]:
import shutil, subprocess

print("bisheng  :", shutil.which("bisheng") or "⚠️  未找到，请重新执行上一个单元格")
print("npu-smi  :", shutil.which("npu-smi") or "⚠️  未找到")
if shutil.which("npu-smi"):
    print()
    print(
        subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout[
            :1800
        ]
    )

## 6. 版本设计总览

四个版本的核函数全部写在同一个 `.asc` 文件中，共用同一份 Host 侧代码、同一组输入与同一套计时逻辑。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 新增的唯一概念 | 算子类 | 输入输出类型 | 中间计算类型 | 每元素搬运量 | 片上占用 |
| --- | --- | --- | --- | --- | --- | --- |
| **v1** | 基础 API 组合与中间缓冲 | `KernelSigmoidBaseApi<float>` | float | float | 8 字节 | 80 KB（42%） |
| **v2** | 高阶 API `Sigmoid` | `KernelSigmoidHighApi` | float | float | 8 字节 | 64 KB（33%） |
| **v3** | 半精度 `half` | `KernelSigmoidBaseApi<half>` | **half** | half | **4 字节** | 40 KB（21%） |
| **v4** | `Cast` 混合精度 | `KernelSigmoidMixed` | half | **float** | 4 字节 | 64 KB（33%） |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">新增的唯一概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">算子类</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">输入输出类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">中间计算类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">每元素搬运量</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">片上占用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础 API 组合与中间缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelSigmoidBaseApi&lt;float&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8 字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">80 KB（42%）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API <code>Sigmoid</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelSigmoidHighApi</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">float</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8 字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">64 KB（33%）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">半精度 <code>half</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelSigmoidBaseApi&lt;half&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>half</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">half</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>4 字节</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">40 KB（21%）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v4</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Cast</code> 混合精度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelSigmoidMixed</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">half</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>float</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4 字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">64 KB（33%）</td>
</tr>
</tbody>
</table>

百分比是相对单个矢量核的 Unified Buffer 容量 192 KB 而言。

**两条线索**：

- **v1 → v2**：**编程效率**。同样的计算、同样的数据类型，四条指令收敛为一次调用。
- **v1 → v3 → v4**：**精度与吞吐**。先用 `half` 换取搬运量减半，再发现中间量溢出，最后用混合精度修正。

这两条线索的对照本身就是一条结论：**高阶 API 决定代码怎么写、数值处理由谁负责；数据类型决定搬运量与可表示的范围。二者相互独立。**

**三个算子类的对应关系**：v1 与 v3 的算子类逐字相同，差别只有数据类型，因此写成一个**类模板** `KernelSigmoidBaseApi<T>`，v1 取 `float`、v3 取 `half`——两个版本的差别被压缩到核函数入口的一个模板实参上。v2 与 v4 各有一个类。

下面的代码按 **Device 侧三个算子类 + 核函数入口** 与 **Host 侧基础设施 + 主程序** 的顺序写入同一个文件。

## 7. Device 侧实现

### 7.1 文件头与参数

先写入头文件与全部可调参数。`TOTAL_LENGTH` 与输入取值范围只是**默认值**，实际取值由命令行参数决定；`TILE_LENGTH` 与 `BLOCK_DIM` 是编译期常量，参数扫描时用 `bisheng -D` 覆盖。

In [ ]:
%%writefile src_sigmoid/ascendc_sigmoid.asc
/**
 * 并行计算 第六章 实验四：激活函数 Sigmoid
 *
 * 本文件包含四个版本的核函数、CPU 基准、数据生成、精度校验与 main，
 * 由一条 bisheng 命令编译为单个可执行程序：
 *
 *   bisheng src_sigmoid/ascendc_sigmoid.asc --npu-arch=dav-2201 -O2 -lm -o src_sigmoid/ascendc_sigmoid
 *
 * 注意 -lm：本实验的 Host 侧用到 std::exp，bisheng 默认不链接数学库，
 * 缺少该选项会在链接阶段报 undefined symbol: expf
 *
 * 用法：
 *   ./ascendc_sigmoid                   四个版本对照，N 与取值范围取默认值
 *   ./ascendc_sigmoid <N>               指定元素总数
 *   ./ascendc_sigmoid <N> <xrange>      指定元素总数与输入取值范围 [-xrange, xrange]
 */
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <cstring>  // 半精度编解码：memcpy
#include <ctime>    // 计时：clock_gettime
#include <vector>

#include "acl/acl.h"          // Host 侧
#include "kernel_operator.h"  // Device 侧

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */

/* 元素总数的默认值：2^21 = 2 097 152 */
#ifndef LAB_TOTAL_LENGTH
#define LAB_TOTAL_LENGTH (2 * 1024 * 1024)
#endif
constexpr uint32_t TOTAL_LENGTH = static_cast<uint32_t>(LAB_TOTAL_LENGTH);

/* 分块长度：单位是元素而非字节。float 版单块 16 KB，half 版单块 8 KB */
#ifndef LAB_TILE_LENGTH
#define LAB_TILE_LENGTH (4 * 1024)
#endif
constexpr uint32_t TILE_LENGTH = static_cast<uint32_t>(LAB_TILE_LENGTH);

/* 参与计算的核数 */
#ifndef LAB_BLOCK_DIM
#define LAB_BLOCK_DIM (8)
#endif
constexpr uint32_t BLOCK_DIM = static_cast<uint32_t>(LAB_BLOCK_DIM);

/* 计时参数：预热次数与重复次数 */
#ifndef LAB_REPEAT
#define LAB_REPEAT (50)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 3;

/* 队列深度：TQue 模板的第二个参数，编译期常量 */
constexpr uint32_t QUEUE_DEPTH = 2;

/* half 的两条边界，见 3 节。写成常量而不是用 std::log 现算，
 * 是为了在源码中直接给出阈值的来源：
 *   X_OVF = -ln(65504)          中间量 e^(-x) 超出 half 上限的位置
 *   X_UNF = ln(2^-24)           sigma(x) 小于 half 最小非规格化数的位置 */
constexpr double HALF_MAX = 65504.0;
constexpr double HALF_MIN_SUBNORMAL = 5.9604644775390625e-8; /* 2^-24 */
constexpr double X_OVF = -11.089866488461016;
constexpr double X_UNF = -16.635532333438686;

/* 绝对误差容差。取值依据见 4.2 节：容差由【本实现链上精度最低的那一环】决定，
 * 而不是由数据类型的机器精度决定。
 *   ATOL_HIGH_API：v2 只调用高阶 Sigmoid，其数值处理由实现负责，
 *                  应当达到 float32 的水平
 *   ATOL_BASE_API：v1/v3/v4 的链上有 Reciprocal——硬件提供的近似求倒数接口，
 *                  实测相对误差在 1e-3 量级，比 half 的输出分辨率（4.9e-4）
 *                  还要粗，因此三个版本的容差都由它决定，与数据类型无关。
 *                  动手练习第 4 题把它换成 Div，届时容差可以收回到 1e-6 附近 */
constexpr double ATOL_HIGH_API = 1e-6;
constexpr double ATOL_BASE_API = 5e-3;

### 7.2 v1 与 v3 共用：基础 API 组合 `KernelSigmoidBaseApi<T>`

这个类模板同时是 v1（`T = float`）与 v3（`T = half`）的实现。**两个版本的差别只有一个模板实参**，计算逻辑、分块方式、流水结构完全相同——这正是本实验能把「数据类型的选择」单独隔离出来考察的前提。

`Compute` 里的四行与 §1 表格中的四步逐条对应。中间量 `t` 用 `TBuf` 而不是 `TQue`，判据与实验三一致：它不跨流水任务。

一种更省内存的写法是把中间结果直接写在输出缓冲 `yLocal` 上，省掉这块 `TBuf`。就地写法在这里可行——`Muls`、`Exp`、`Adds`、`Reciprocal` 这几个**基础**算术接口并未禁止源操作数与目的操作数地址重叠。本版仍保留独立的中间缓冲，目的是：让 t 的四步变换逐一可见。

**片上缓冲占用**：输入队列 `TILE_LENGTH × sizeof(T) × 2` + 输出队列同上 + 中间缓冲 `TILE_LENGTH × sizeof(T)`。`T = float`、`TILE_LENGTH = 4096` 时为 32 + 32 + 16 = 80 KB，占 UB（192 KB）的 42%；`T = half` 时减半为 40 KB。

In [ ]:
%%writefile -a src_sigmoid/ascendc_sigmoid.asc
/* ============ v1 与 v3 共用：基础 API 组合 ============
 * 新增概念：复合函数的基础 API 组合与中间缓冲
 *   y = 1 / (1 + e^(-x)) 分解为四条基础矢量指令：
 *     Muls(t, x, -1) -> Exp(t, t) -> Adds(t, t, 1) -> Reciprocal(y, t)
 *   中间量 t 既非输入也非输出，也不跨流水任务，故用 VECCALC 位置的 TBuf。
 *
 * v1 取 T = float，v3 取 T = half —— 两个版本仅此一处差别。
 */
template <typename T>
class KernelSigmoidBaseApi {
 public:
  __aicore__ inline KernelSigmoidBaseApi() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t n) {
    /* 核间切分：与实验二、实验三相同，由 blockIdx 算出偏移 */
    blockLength_ = n / AscendC::GetBlockNum();
    tileNum_ = blockLength_ / TILE_LENGTH;
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ T *>(x) + offset, blockLength_);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ T *>(y) + offset, blockLength_);

    pipe.InitBuffer(inQueueX, QUEUE_DEPTH, TILE_LENGTH * sizeof(T));
    pipe.InitBuffer(outQueueY, QUEUE_DEPTH, TILE_LENGTH * sizeof(T));
    /* 中间量缓冲：只在 Compute 内部使用，不跨流水任务，故用 TBuf */
    pipe.InitBuffer(tmpBuf, TILE_LENGTH * sizeof(T));
  }

  __aicore__ inline void Process() {
    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Compute();
      CopyOut(i);
    }
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<T> xLocal = inQueueX.AllocTensor<T>();
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(xLocal);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<T> xLocal = inQueueX.DeQue<T>();
    AscendC::LocalTensor<T> yLocal = outQueueY.AllocTensor<T>();
    AscendC::LocalTensor<T> t = tmpBuf.Get<T>();

    /* 与 1 节表格中的四步逐条对应。T = half 时，第二步的结果会在
     * x < -11.09 处超出 half 的表示上限——这是 v3 失效的全部原因 */
    AscendC::Muls(t, xLocal, static_cast<T>(-1), TILE_LENGTH);
    AscendC::Exp(t, t, TILE_LENGTH);
    AscendC::Adds(t, t, static_cast<T>(1), TILE_LENGTH);
    AscendC::Reciprocal(yLocal, t, TILE_LENGTH);

    outQueueY.EnQue(yLocal);
    inQueueX.FreeTensor(xLocal);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<T> yLocal = outQueueY.DeQue<T>();
    AscendC::DataCopy(yGm[progress * TILE_LENGTH], yLocal, TILE_LENGTH);
    outQueueY.FreeTensor(yLocal);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueueY;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf; /* 中间量 t */
  AscendC::GlobalTensor<T> xGm, yGm;
  uint32_t blockLength_ = 0;
  uint32_t tileNum_ = 0;
};

### 7.3 v2：高阶 API `KernelSigmoidHighApi`

本版与 v1（`T = float`）的差异只有两处：`Compute` 中的四条基础 API 换成一次 `AscendC::Sigmoid` 调用，中间缓冲 `TBuf` 随之取消。取消 `TBuf` 之后片上占用降到 64 KB。

In [ ]:
%%writefile -a src_sigmoid/ascendc_sigmoid.asc
/* ============ v2：高阶 API ============
 * 新增概念：一次调用替代整条基础 API 链
 *   计算逻辑与 v1 完全等价，中间缓冲由接口框架内部管理，
 *   因此本类不再申请 TBuf——但必须给框架留出未分配的片上空间。
 *
 * 约束：高阶 Sigmoid 不支持源、目的与临时空间互相重叠。
 *       本版的 xLocal 来自输入队列、yLocal 来自输出队列，天然不重叠。
 */
class KernelSigmoidHighApi {
 public:
  __aicore__ inline KernelSigmoidHighApi() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t n) {
    blockLength_ = n / AscendC::GetBlockNum();
    tileNum_ = blockLength_ / TILE_LENGTH;
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(x) + offset,
                        blockLength_);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(y) + offset,
                        blockLength_);

    /* 只有两个队列，合计 64 KB，UB 中仍有充足余量供框架申请临时空间 */
    pipe.InitBuffer(inQueueX, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
    pipe.InitBuffer(outQueueY, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
  }

  __aicore__ inline void Process() {
    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Compute();
      CopyOut(i);
    }
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(xLocal);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = outQueueY.AllocTensor<float>();

    /* 整条链收敛为一次调用 */
    AscendC::Sigmoid(yLocal, xLocal, TILE_LENGTH);

    outQueueY.EnQue(yLocal);
    inQueueX.FreeTensor(xLocal);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<float> yLocal = outQueueY.DeQue<float>();
    AscendC::DataCopy(yGm[progress * TILE_LENGTH], yLocal, TILE_LENGTH);
    outQueueY.FreeTensor(yLocal);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueueY;
  AscendC::GlobalTensor<float> xGm, yGm;
  uint32_t blockLength_ = 0;
  uint32_t tileNum_ = 0;
};

### 7.4 v4：混合精度 `KernelSigmoidMixed`

v3 的问题出在中间量而不是输入输出。因此对策是：**输入输出保持 `half` 以保留搬运量减半的收益，中间计算转到 `float` 域完成。**

`Compute` 的形状因此变成「升精度 — 计算 — 降精度」三步：

```cpp
Cast(src, xLocal, RoundMode::CAST_NONE, TILE_LENGTH);   // half -> float
// ... 在 float 域内完成 1 节表格中的四步 ...
Cast(yLocal, src, RoundMode::CAST_NONE, TILE_LENGTH);   // float -> half
```

**关于取整模式**：官方对 `CAST_NONE` 的定义是——**在转换有精度损失时使用 `CAST_RINT` 模式，在不涉及精度损失时不进行舍入**；而 `CAST_RINT` 即四舍六入五成双。因此这里两次调用都写 `CAST_NONE` 是正确的：`half` 转 `float` 无精度损失、不舍入，`float` 转 `half` 有精度损失、自动按四舍六入五成双处理。

> **官方样例的写法略有不同**：在 `bfloat16_t` 输入的 Add 算子中，官方在升精度处写 `CAST_NONE`、在降精度处**显式写 `CAST_RINT`**。按上面的定义两者等价，但显式写出取整模式更能表明意图，工程代码中建议照此办理。

<img src="images/06.04_mixed_precision_flow.png" alt="06.04_mixed_precision_flow" width="820px">

*输入为 bfloat16_t 类型的 Add 计算流程。官方的这条流水与本版的形状完全一致：搬入后先 `Cast` 升精度，在 `float` 域完成计算，再 `Cast` 降精度后搬出*

**代价**：需要两块 `float` 中间缓冲，片上占用为「队列 2 × 8 KB × 2 + 中间缓冲 2 × 16 KB = 64 KB」，高于 v3 的 40 KB，但仍低于 v1 的 80 KB；此外多出两次 `Cast` 的计算开销。由于本算子受访存制约，这些额外的片上计算是否会被搬运时间掩盖，请以实测数据判断。

In [ ]:
%%writefile -a src_sigmoid/ascendc_sigmoid.asc
/* ============ v4：混合精度 ============
 * 新增概念：Cast 与混合精度
 *   输入输出保持 half 以保留搬运量减半的收益；
 *   中间计算通过 Cast 转到 float 域，e^(-x) 因而不会溢出。
 *
 * 取整模式：half -> float 无精度损失，取 CAST_NONE；
 *           float -> half 有精度损失，CAST_NONE 表示四舍六入五成双。
 */
class KernelSigmoidMixed {
 public:
  __aicore__ inline KernelSigmoidMixed() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t n) {
    blockLength_ = n / AscendC::GetBlockNum();
    tileNum_ = blockLength_ / TILE_LENGTH;
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ half *>(x) + offset,
                        blockLength_);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ half *>(y) + offset,
                        blockLength_);

    pipe.InitBuffer(inQueueX, QUEUE_DEPTH, TILE_LENGTH * sizeof(half));
    pipe.InitBuffer(outQueueY, QUEUE_DEPTH, TILE_LENGTH * sizeof(half));
    /* 两块 float 中间缓冲：一块存放转换后的输入与最终结果，一块存放中间量 */
    pipe.InitBuffer(srcF32Buf, TILE_LENGTH * sizeof(float));
    pipe.InitBuffer(tmpF32Buf, TILE_LENGTH * sizeof(float));
  }

  __aicore__ inline void Process() {
    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Compute();
      CopyOut(i);
    }
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<half> xLocal = inQueueX.AllocTensor<half>();
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(xLocal);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<half> xLocal = inQueueX.DeQue<half>();
    AscendC::LocalTensor<half> yLocal = outQueueY.AllocTensor<half>();
    AscendC::LocalTensor<float> src = srcF32Buf.Get<float>();
    AscendC::LocalTensor<float> t = tmpF32Buf.Get<float>();

    /* half -> float：无精度损失 */
    AscendC::Cast(src, xLocal, AscendC::RoundMode::CAST_NONE, TILE_LENGTH);

    /* 与 v1 逐行相同，只是这里在 float 域内进行，中间量不会溢出 */
    AscendC::Muls(t, src, static_cast<float>(-1), TILE_LENGTH);
    AscendC::Exp(t, t, TILE_LENGTH);
    AscendC::Adds(t, t, static_cast<float>(1), TILE_LENGTH);
    AscendC::Reciprocal(src, t, TILE_LENGTH);

    /* float -> half：按四舍六入五成双舍入 */
    AscendC::Cast(yLocal, src, AscendC::RoundMode::CAST_NONE, TILE_LENGTH);

    outQueueY.EnQue(yLocal);
    inQueueX.FreeTensor(xLocal);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<half> yLocal = outQueueY.DeQue<half>();
    AscendC::DataCopy(yGm[progress * TILE_LENGTH], yLocal, TILE_LENGTH);
    outQueueY.FreeTensor(yLocal);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueueY;
  AscendC::TBuf<AscendC::TPosition::VECCALC> srcF32Buf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpF32Buf;
  AscendC::GlobalTensor<half> xGm, yGm;
  uint32_t blockLength_ = 0;
  uint32_t tileNum_ = 0;
};

### 7.5 四个核函数入口

每个入口的第一行都是 `KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);`，含义与前两个实验相同：声明本核函数只使用矢量核，使 `GetBlockNum()` 等于启动时指定的 `blockDim`。

请特别注意 `sigmoid_v1` 与 `sigmoid_v3` 这两个入口：**它们的差别只有一个模板实参**。这是本实验能把「数据类型的选择」当作一个独立变量来考察的技术前提。

In [ ]:
%%writefile -a src_sigmoid/ascendc_sigmoid.asc
/* ===================== 核函数入口 ===================== */

extern "C" __global__ __aicore__ void sigmoid_v1(GM_ADDR x, GM_ADDR y,
                                                 uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY); /* 声明为纯矢量内核 */
  KernelSigmoidBaseApi<float> op;
  op.Init(x, y, n);
  op.Process();
}

extern "C" __global__ __aicore__ void sigmoid_v2(GM_ADDR x, GM_ADDR y,
                                                 uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelSigmoidHighApi op;
  op.Init(x, y, n);
  op.Process();
}

extern "C" __global__ __aicore__ void sigmoid_v3(GM_ADDR x, GM_ADDR y,
                                                 uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelSigmoidBaseApi<half> op; /* 与 v1 的唯一差别：模板实参 */
  op.Init(x, y, n);
  op.Process();
}

extern "C" __global__ __aicore__ void sigmoid_v4(GM_ADDR x, GM_ADDR y,
                                                 uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelSigmoidMixed op;
  op.Init(x, y, n);
  op.Process();
}

## 8. Host 侧实现

Device 侧到此结束。Host 侧的代码分两部分写入：先是一组基础设施，然后是主程序。

### 8.1 基础设施：半精度编解码、数据生成、两套真值、双口径校验与分段统计

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 内容 | 说明 |
| --- | --- |
| `ACL_CHECK`、`GetTimeMs`、`NpuInit` / `NpuFinalize` | 与前三个实验相同 |
| `F32ToF16` / `F16ToF32` | 半精度编解码，纯整数位运算，不依赖任何库 |
| `LcgNextFloat` / `GenerateInput` | 固定种子的线性同余发生器，生成 [−xrange, xrange] 上的均匀分布 |
| `SigmoidRef` | 参考真值，以 `double` 计算 |
| `RunOnCpu` | CPU 单线程 `float32` 基准，结果写入实际数组并参与后续统计，不会被编译器消除 |
| `Verify` | **绝对误差**判定 `PASS` / `FAIL`，同时报告最大相对误差 |
| `ReportError` | 按 §3 的三段区间统计相对误差，并输出用于作图的分箱数据 |
| `TIME_KERNEL` | 与前三个实验相同：先预热，再重复下发并逐次同步后取平均 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_CHECK</code>、<code>GetTimeMs</code>、<code>NpuInit</code> / <code>NpuFinalize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与前三个实验相同</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>F32ToF16</code> / <code>F16ToF32</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">半精度编解码，纯整数位运算，不依赖任何库</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>LcgNextFloat</code> / <code>GenerateInput</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">固定种子的线性同余发生器，生成 [−xrange, xrange] 上的均匀分布</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>SigmoidRef</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考真值，以 <code>double</code> 计算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RunOnCpu</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 单线程 <code>float32</code> 基准，结果写入实际数组并参与后续统计，不会被编译器消除</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Verify</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>绝对误差</strong>判定 <code>PASS</code> / <code>FAIL</code>，同时报告最大相对误差</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReportError</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按 §3 的三段区间统计相对误差，并输出用于作图的分箱数据</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TIME_KERNEL</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与前三个实验相同：先预热，再重复下发并逐次同步后取平均</td>
</tr>
</tbody>
</table>

关于其中三点的说明：

**为什么要自己写半精度编解码。** Host 侧的标准 C++ 没有 `half` 类型，而本实验必须在 Host 侧生成 `half` 输入、并把 `half` 输出解码回来才能校验。用整数位运算实现这两个函数只需二十余行，且有一个额外的好处：`half` 的编码规则——5 位指数、10 位尾数、非规格化数的处理——在源码里全部显式可见。

**为什么 `half` 版本要用另一份真值。** 见 §4.1：`half` 版本的输入是量化之后的值，用 `float` 输入算出的真值去校验它，会把输入量化的误差记到算子头上。代码中 `goldenH` 正是由 `F16ToF32(F32ToF16(x))` 算出的。

**为什么 `ReportError` 要同时输出分段统计与分箱数据。** 分段统计（`[SEG]` 行）用于判定，其中 `over_half` 一列直接给出 §3.1 那条判据的结果；分箱数据（`[ERRBIN]` 行）用于作图，Notebook 中直接解析这一行绘制误差随 x 的分布。两者服务于不同的目的：前者要的是结论，后者要的是形状。

程序运行时会打印四类可被程序解析的记录行：

```
[BASE]   n=2097152 xrange=20.000 cpu_ms=... cpu_max_abs_err=... cpu_max_rel_err=...
[VERIFY] ver=v3 n=2097152 dtype=f16 atol=5.0e-03 max_abs_err=... max_rel_err=... result=PASS
[PERF]   ver=v3 n=2097152 blockDim=8 bytes_per_elem=4 ub_kb=40.0 cpu_ms=... kernel_ms=... sp_kernel=...
[SEG]    ver=v3 seg=2 x_lo=-16.636 x_hi=-11.090 count=... max_rel=... over_half=...
[ERRBIN] ver=v3 nbins=160 x_lo=-20.000 x_hi=20.000 rel=1.2e-03,3.4e-03,...
```

> 上面五行是**格式示例**，其中的数值不代表实测结果。

In [ ]:
%%writefile -a src_sigmoid/ascendc_sigmoid.asc
/* ============================================================
 *                       Host 侧代码
 * ============================================================ */

#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

static inline double GetTimeMs() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1000000.0;
}

static int32_t g_deviceId = 0;
static aclrtStream g_stream = nullptr;

static void NpuInit() {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(g_deviceId));
  ACL_CHECK(aclrtCreateStream(&g_stream));
}

static void NpuFinalize() {
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(g_deviceId));
  ACL_CHECK(aclFinalize());
}

/* ---------- 半精度编解码：纯整数位运算，不依赖任何库 ----------
 * half 的位域：1 位符号 + 5 位指数（偏移 15）+ 10 位尾数
 * float 的位域：1 位符号 + 8 位指数（偏移 127）+ 23 位尾数            */
static uint16_t F32ToF16(float f) {
  uint32_t u;
  std::memcpy(&u, &f, sizeof(u));
  const uint32_t sign = (u >> 16) & 0x8000u;
  const uint32_t rawExp = (u >> 23) & 0xFFu;
  uint32_t man = u & 0x7FFFFFu;

  if (rawExp == 0xFFu) { /* Inf 或 NaN */
    return static_cast<uint16_t>(sign | 0x7C00u | (man ? 0x200u : 0u));
  }
  const int32_t exp = static_cast<int32_t>(rawExp) - 127 + 15;
  if (exp >= 0x1F) { /* 超出 half 上限：饱和为 Inf */
    return static_cast<uint16_t>(sign | 0x7C00u);
  }
  if (exp <= 0) { /* 落入非规格化数区间，或彻底下溢为 0 */
    if (exp < -10) {
      return static_cast<uint16_t>(sign);
    }
    man |= 0x800000u; /* 补上规格化数隐含的最高位 1 */
    const uint32_t shift = static_cast<uint32_t>(14 - exp);
    uint32_t halfMan = man >> shift;
    if ((man >> (shift - 1)) & 1u) {
      ++halfMan; /* 就近舍入 */
    }
    return static_cast<uint16_t>(sign | halfMan);
  }
  uint32_t halfMan = man >> 13;
  if ((man >> 12) & 1u) {
    ++halfMan; /* 就近舍入 */
  }
  /* 这里必须用加法而不是按位或：舍入可能把 halfMan 推到 0x400（10 位放不下），
   * 此时进位应当传到指数位上去。若写成 `| halfMan`，进位会被丢掉，
   * 结果会退回本区间的下界——例如 0.4999 会被编码成 0.25。思考题第 9 题即此 */
  return static_cast<uint16_t>(sign +
                               ((static_cast<uint32_t>(exp) << 10) + halfMan));
}

static float F16ToF32(uint16_t h) {
  const uint32_t sign = static_cast<uint32_t>(h & 0x8000u) << 16;
  const uint32_t exp = (h >> 10) & 0x1Fu;
  uint32_t man = h & 0x3FFu;
  uint32_t u;

  if (exp == 0) {
    if (man == 0) {
      u = sign; /* 正负零 */
    } else { /* 非规格化数：左移到规格化位置，指数相应减小 */
      int32_t e = -1;
      do {
        ++e;
        man <<= 1;
      } while (!(man & 0x400u));
      man &= 0x3FFu;
      u = sign | (static_cast<uint32_t>(127 - 15 - e) << 23) | (man << 13);
    }
  } else if (exp == 0x1F) {
    u = sign | 0x7F800000u | (man << 13); /* Inf 或 NaN */
  } else {
    u = sign | ((exp + 127 - 15) << 23) | (man << 13);
  }
  float f;
  std::memcpy(&f, &u, sizeof(f));
  return f;
}

/* ---------- 数据生成：线性同余发生器，输出 [-range, range] ---------- */
static inline float LcgNextFloat(uint32_t &state) {
  state = state * 1664525u + 1013904223u;
  return static_cast<float>(state >> 8) * (2.0f / 16777216.0f) - 1.0f;
}

static void GenerateInput(std::vector<float> &x, float range, uint32_t seed) {
  uint32_t s = seed;
  for (size_t i = 0; i < x.size(); ++i) {
    x[i] = LcgNextFloat(s) * range;
  }
}

/* ---------- 参考真值：以 double 计算 ---------- */
static inline double SigmoidRef(double v) { return 1.0 / (1.0 + std::exp(-v)); }

/* ---------- CPU 单线程 float32 基准 ----------
 * 结果写入实际数组，随后参与误差统计，因此不会被编译器消除 */
static double RunOnCpu(const std::vector<float> &x, std::vector<float> &y,
                       int warmup, int repeat) {
  const size_t n = x.size();
  for (int r = 0; r < warmup; ++r) {
    for (size_t i = 0; i < n; ++i) y[i] = 1.0f / (1.0f + std::exp(-x[i]));
  }
  const double t0 = GetTimeMs();
  for (int r = 0; r < repeat; ++r) {
    for (size_t i = 0; i < n; ++i) y[i] = 1.0f / (1.0f + std::exp(-x[i]));
  }
  return (GetTimeMs() - t0) / repeat;
}

/* ---------- 双口径校验：绝对误差判定，相对误差一并报告 ---------- */
static bool Verify(const char *ver, const char *dtype, uint32_t n,
                   const std::vector<float> &out,
                   const std::vector<double> &golden, double atol) {
  double maxAbs = 0.0, maxRel = 0.0;
  for (size_t i = 0; i < golden.size(); ++i) {
    const double g = golden[i];
    const double a = std::fabs(static_cast<double>(out[i]) - g);
    if (a > maxAbs) maxAbs = a;
    const double r = a / g; /* sigma(x) 恒大于 0，不必保护分母 */
    if (r > maxRel) maxRel = r;
  }
  const bool ok = (maxAbs <= atol);
  std::printf(
      "[VERIFY] ver=%s n=%u dtype=%s atol=%.1e max_abs_err=%.3e "
      "max_rel_err=%.3e result=%s\n",
      ver, n, dtype, atol, maxAbs, maxRel, ok ? "PASS" : "FAIL");
  return ok;
}

/* ---------- 分段统计与分箱数据 ----------
 * 分段：按 3 节的两条阈值把输入切成三段，逐段统计最大相对误差。
 *       传入的 x 是【算子实际看到的输入值】——half 版本传的是量化之后的值，
 *       否则紧贴阈值的少数元素会被分到错误的区间里，
 *       并数出「相对误差超过 1/2」的元素个数——即 3.1 节那条判据；
 * 分箱：把 [-range, range] 均分为 NBINS 个箱，逐箱取最大相对误差，供作图用 */
constexpr int32_t NBINS = 160;

static void ReportError(const char *ver, const std::vector<float> &x,
                        const std::vector<float> &out,
                        const std::vector<double> &golden, float range) {
  const double edge[4] = {-static_cast<double>(range), X_UNF, X_OVF,
                          static_cast<double>(range)};
  double segMax[3] = {0.0, 0.0, 0.0};
  int64_t segCnt[3] = {0, 0, 0}, segOver[3] = {0, 0, 0};
  std::vector<double> binMax(NBINS, 0.0);
  const double lo = -static_cast<double>(range);
  const double step = 2.0 * static_cast<double>(range) / NBINS;

  for (size_t i = 0; i < golden.size(); ++i) {
    const double xv = static_cast<double>(x[i]);
    const double rel =
        std::fabs(static_cast<double>(out[i]) - golden[i]) / golden[i];

    int s = 2;
    if (xv < edge[1]) {
      s = 0;
    } else if (xv < edge[2]) {
      s = 1;
    }
    ++segCnt[s];
    if (rel > segMax[s]) segMax[s] = rel;
    if (rel > 0.5) ++segOver[s];

    int b = static_cast<int>((xv - lo) / step);
    if (b < 0) b = 0;
    if (b >= NBINS) b = NBINS - 1;
    if (rel > binMax[b]) binMax[b] = rel;
  }

  for (int s = 0; s < 3; ++s) {
    /* 取值范围小于 11.09 时前两段为空。此处如实报告 count=0、不做除法，
     * 打印的区间端点也钳到实际取值范围内，Notebook 端据此跳过空区间 */
    double a = edge[s] < lo ? lo : edge[s];
    double b = edge[s + 1] > -lo ? -lo : edge[s + 1];
    if (b < a) b = a;
    std::printf(
        "[SEG]    ver=%s seg=%d x_lo=%.3f x_hi=%.3f count=%lld max_rel=%.3e "
        "over_half=%lld\n",
        ver, s + 1, a, b, static_cast<long long>(segCnt[s]), segMax[s],
        static_cast<long long>(segOver[s]));
  }

  std::printf("[ERRBIN] ver=%s nbins=%d x_lo=%.3f x_hi=%.3f rel=", ver, NBINS,
              lo, -lo);
  for (int b = 0; b < NBINS; ++b) {
    std::printf("%s%.3e", b ? "," : "", binMax[b]);
  }
  std::printf("\n");
}

/* ---------- 打印一行可被程序解析的性能记录 ----------
 * 加速比与其分子、分母一并输出：脱离基准耗时的加速比无法解读（见实验二 13 节第 5 条）。 */
static void ReportPerf(const char *ver, uint32_t n, uint32_t bytesPerElem,
                       double ubKB, double cpuMs, double kernelMs) {
  std::printf(
      "[PERF]   ver=%s n=%u blockDim=%u bytes_per_elem=%u ub_kb=%.1f "
      "cpu_ms=%.4f kernel_ms=%.4f sp_kernel=%.4f\n",
      ver, n, BLOCK_DIM, bytesPerElem, ubKB, cpuMs, kernelMs,
      cpuMs / kernelMs);
}

/* ---------- 计时宏：核函数耗时（不含主机与设备之间的数据搬运） ---------- */
#define TIME_KERNEL(LAUNCH, OUT_MS)                                               \
  do {                                                                            \
    for (int _w = 0; _w < WARMUP; ++_w) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream));                                \
    }                                                                             \
    const double _t0 = GetTimeMs();                                               \
    for (int _r = 0; _r < REPEAT; ++_r) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream)); /* 先同步再停止计时 */ \
    }                                                                             \
    (OUT_MS) = (GetTimeMs() - _t0) / REPEAT;                                      \
  } while (0)


### 8.2 主程序

主程序的结构与前三个实验一致，本实验特有的部分集中在数据准备这一段：同一批随机数要以两种格式各存一份，并各配一份参考真值。

`RUN_VERSION` 宏把每个版本的固定流程收成一处：运行一次并校验 → 分段统计误差 → 计时。四个版本的差别只在于用哪一对设备缓冲、哪一份真值、以及每元素多少字节。

In [ ]:
%%writefile -a src_sigmoid/ascendc_sigmoid.asc
/* 每个版本的固定流程：运行一次并校验 -> 分段统计 -> 计时 */
#define RUN_VERSION(TAG, DTYPE, LAUNCH, YDEV, HOSTY, BYTES,                 \
                    DECODE, XARR, GOLDEN, ATOL, BPE, UBKB)                  \
  do {                                                                      \
    LAUNCH;                                                                 \
    ACL_CHECK(aclrtSynchronizeStream(g_stream));                            \
    ACL_CHECK(                                                              \
        aclrtMemcpy(HOSTY, BYTES, YDEV, BYTES, ACL_MEMCPY_DEVICE_TO_HOST)); \
    DECODE;                                                                 \
    allPass &= Verify(TAG, DTYPE, n, dec, GOLDEN, ATOL);                    \
    ReportError(TAG, XARR, dec, GOLDEN, xRange);                            \
    TIME_KERNEL(LAUNCH, kMs);                                               \
    ReportPerf(TAG, n, BPE, UBKB, cpuMs, kMs);                              \
  } while (0)

int32_t main(int argc, char *argv[]) {
  /* ---------- 解析命令行 ---------- */
  uint32_t n = TOTAL_LENGTH;
  float xRange = 20.0f;
  if (argc > 1) n = static_cast<uint32_t>(std::strtoul(argv[1], nullptr, 10));
  if (argc > 2) xRange = static_cast<float>(std::atof(argv[2]));

  const uint32_t granularity = BLOCK_DIM * TILE_LENGTH;
  if (n == 0 || n % granularity != 0) {
    std::printf(
        "[FATAL] N 必须是 BLOCK_DIM x TILE_LENGTH = %u 的整数倍，当前 N = %u\n",
        granularity, n);
    return 1;
  }
  if (!(xRange > 0.0f)) {
    std::printf("[FATAL] 取值范围必须为正数，当前为 %f\n", xRange);
    return 1;
  }

  /* ---------- 数据准备：同一批随机数存两种格式，各配一份真值 ---------- */
  std::vector<float> x32(n), xh(n), dec(n), cpuOut(n);
  std::vector<uint16_t> x16(n), y16(n);
  std::vector<float> y32(n);
  std::vector<double> golden32(n), goldenH(n);

  GenerateInput(x32, xRange, 2026u);
  for (uint32_t i = 0; i < n; ++i) {
    x16[i] = F32ToF16(x32[i]);
    xh[i] = F16ToF32(x16[i]); /* half 版本实际看到的输入值 */
    golden32[i] = SigmoidRef(static_cast<double>(x32[i]));
    goldenH[i] = SigmoidRef(static_cast<double>(xh[i]));
  }

  /* ---------- CPU 基准；同时报告其自身相对 double 真值的误差 ---------- */
  const int cpuRepeat = (REPEAT <= 1) ? 1 : ((n > (4u << 20)) ? 3 : 10);
  const double cpuMs =
      RunOnCpu(x32, cpuOut, (REPEAT <= 1) ? 1 : WARMUP, cpuRepeat);
  double cpuAbs = 0.0, cpuRel = 0.0;
  for (uint32_t i = 0; i < n; ++i) {
    const double a = std::fabs(static_cast<double>(cpuOut[i]) - golden32[i]);
    if (a > cpuAbs) cpuAbs = a;
    const double r = a / golden32[i];
    if (r > cpuRel) cpuRel = r;
  }

  std::printf("N=%u  TILE_LENGTH=%u  BLOCK_DIM=%u  REPEAT=%d  X_RANGE=%.3f\n",
              n, TILE_LENGTH, BLOCK_DIM, REPEAT, xRange);
  std::printf(
      "[BASE]   n=%u xrange=%.3f cpu_ms=%.4f cpu_max_abs_err=%.3e "
      "cpu_max_rel_err=%.3e x_ovf=%.4f x_unf=%.4f half_max=%.0f "
      "half_min_sub=%.3e\n",
      n, xRange, cpuMs, cpuAbs, cpuRel, X_OVF, X_UNF, HALF_MAX,
      HALF_MIN_SUBNORMAL);

  /* ---------- 申请显存并搬入两种格式的输入 ---------- */
  NpuInit();
  const size_t bytes32 = static_cast<size_t>(n) * sizeof(float);
  const size_t bytes16 = static_cast<size_t>(n) * sizeof(uint16_t);
  uint8_t *xDev32 = nullptr, *yDev32 = nullptr;
  uint8_t *xDev16 = nullptr, *yDev16 = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&xDev32, bytes32, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&yDev32, bytes32, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&xDev16, bytes16, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&yDev16, bytes16, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMemcpy(xDev32, bytes32, x32.data(), bytes32,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  ACL_CHECK(aclrtMemcpy(xDev16, bytes16, x16.data(), bytes16,
                        ACL_MEMCPY_HOST_TO_DEVICE));

  bool allPass = true;
  double kMs = 0.0;

/* float 版本的输出可直接使用；half 版本的输出须先解码回 float 再校验 */
#define DECODE_F32 dec.assign(y32.begin(), y32.end())
#define DECODE_F16                   \
  for (uint32_t i = 0; i < n; ++i) { \
    dec[i] = F16ToF32(y16[i]);       \
  }

  /* ---------------- v1：基础 API 组合（float） ---------------- */
  RUN_VERSION("v1", "f32",
              (sigmoid_v1<<<BLOCK_DIM, nullptr, g_stream>>>(xDev32, yDev32, n)),
              yDev32, y32.data(), bytes32, DECODE_F32, x32,
              golden32, ATOL_BASE_API, 8, 80.0);

  /* ---------------- v2：高阶 API（float） ---------------- */
  RUN_VERSION("v2", "f32",
              (sigmoid_v2<<<BLOCK_DIM, nullptr, g_stream>>>(xDev32, yDev32, n)),
              yDev32, y32.data(), bytes32, DECODE_F32, x32,
              golden32, ATOL_HIGH_API, 8, 64.0);

  /* ---------------- v3：基础 API 组合（half） ---------------- */
  RUN_VERSION("v3", "f16",
              (sigmoid_v3<<<BLOCK_DIM, nullptr, g_stream>>>(xDev16, yDev16, n)),
              yDev16, y16.data(), bytes16, DECODE_F16, xh,
              goldenH, ATOL_BASE_API, 4, 40.0);

  /* ---------------- v4：混合精度（half io + float 计算） ---------------- */
  RUN_VERSION("v4", "f16",
              (sigmoid_v4<<<BLOCK_DIM, nullptr, g_stream>>>(xDev16, yDev16, n)),
              yDev16, y16.data(), bytes16, DECODE_F16, xh,
              goldenH, ATOL_BASE_API, 4, 64.0);

  /* ---------- 释放资源（与申请严格成对，顺序相反） ---------- */
  ACL_CHECK(aclrtFree(yDev16));
  ACL_CHECK(aclrtFree(xDev16));
  ACL_CHECK(aclrtFree(yDev32));
  ACL_CHECK(aclrtFree(xDev32));
  NpuFinalize();

  std::printf(allPass ? "[SUCCESS] 全部版本通过绝对误差校验。\n"
                      : "[FAILED] 存在未通过绝对误差校验的版本！\n");
  return allPass ? 0 : 1;
}

## 9. 编译与运行

`-O2` 不能省略——CPU 基准与 NPU 代码在同一次编译中生成，使用 `-O0` 会人为放大基准实现的耗时，使加速比失真。

`-lm` 也不能省略：本实验的 Host 侧用到 `std::exp`，而 `bisheng` 默认不链接数学库，缺少该选项会在链接阶段报 `undefined symbol: expf`。

In [ ]:
import subprocess

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按实验一 §7.2 的表修改
SRC = "src_sigmoid/ascendc_sigmoid.asc"
EXE = "src_sigmoid/ascendc_sigmoid"

# 编译选项集中定义一处，§12 的参数扫描直接复用，避免两处不一致
# -lm：Host 侧用到 std::exp，bisheng 默认不链接数学库
FLAGS = ["--npu-arch=" + ARCH, "-O2", "-lm"]

# bisheng [算子源文件] [编译选项] -o [输出产物名称]
cmd = ["bisheng", SRC] + FLAGS + ["-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

一次运行即可输出四个版本的校验结果、性能记录与误差分布。`[ERRBIN]` 行很长（每行 160 个数值），因此下面只打印其余部分，完整文本保存在 `out_main` 中供后续解析。

In [ ]:
def run_demo(args=(), exe=None, timeout=900):
    # 与实验二、三同名同约定：只返回 stdout；exe 缺省为 §9 编译出的主程序
    proc = subprocess.run(
        ["./" + (exe or EXE)] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0 and not proc.stdout:
        print("返回码", proc.returncode)
        print(proc.stderr)
    return proc.stdout


out_main = run_demo()
print("\n".join(l for l in out_main.splitlines() if not l.startswith("[ERRBIN]")))

### 9.1 解析输出

四类记录行的字段名固定，按空格切分即可解析为字典。`[ERRBIN]` 行的 `rel=` 字段是逗号分隔的一串数值，单独取出。

In [ ]:
def parse_rows(text, tag):
    # 把所有以 [tag] 开头的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if line.startswith("[" + tag + "]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = v if k in ("ver", "result", "dtype") else float(v)
            rows.append(d)
    return rows


def parse_perf(text):
    return parse_rows(text, "PERF")


def parse_verify(text):
    return {r["ver"]: r for r in parse_rows(text, "VERIFY")}


def parse_base(text):
    rows = parse_rows(text, "BASE")
    return rows[0] if rows else None


def parse_errbin(text):
    # [ERRBIN] 的 rel= 字段是逗号分隔的一串数值，不能按空格拆
    out = {}
    for line in text.splitlines():
        if line.startswith("[ERRBIN]"):
            head, rel = line.split("rel=", 1)
            d = dict(kv.split("=", 1) for kv in head.split()[1:])
            out[d["ver"]] = (
                float(d["x_lo"]),
                float(d["x_hi"]),
                [float(v) for v in rel.split(",")],
            )
    return out


rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)
base_main = parse_base(out_main)
seg_main = parse_rows(out_main, "SEG")
bins_main = parse_errbin(out_main)
base_v1 = rows_main[0]["kernel_ms"] if rows_main else 1.0

print(
    "CPU float32 基准：%.4f ms   相对 double 真值的最大绝对误差 %.3e   最大相对误差 %.3e"
    % (base_main["cpu_ms"], base_main["cpu_max_abs_err"], base_main["cpu_max_rel_err"])
)
print(
    "half 的两条阈值：中间量溢出 x < %.2f    输出完全下溢 x < %.2f"
    % (base_main["x_ovf"], base_main["x_unf"])
)
print()

hdr = (
    "版本",
    "类型",
    "字节/元素",
    "UB(KB)",
    "CPU(ms)",
    "kernel(ms)",
    "vs CPU",
    "vs v1",
    "最大绝对误差",
    "最大相对误差",
    "判定",
)
print("%-5s %5s %10s %8s %10s %11s %9s %8s %13s %13s %6s" % hdr)
print("-" * 116)
for r in rows_main:
    v = chk_main[r["ver"]]
    print(
        "%-5s %5s %10d %8.1f %10.4f %11.4f %8.2fx %7.2fx %13.3e %13.3e %6s"
        % (
            r["ver"],
            v["dtype"],
            r["bytes_per_elem"],
            r["ub_kb"],
            r["cpu_ms"],
            r["kernel_ms"],
            r["sp_kernel"],
            base_v1 / r["kernel_ms"],
            v["max_abs_err"],
            v["max_rel_err"],
            v["result"],
        )
    )

## 10. 结果可视化

两张加速比图回答两个不同的问题：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 图 | 基线 | 回答的问题 |
| --- | --- | --- |
| 图一 | **CPU 单线程 float32** | 使用 NPU 的整体收益 |
| 图二 | **v1（float，基础 API 组合）** | 每一项改动各自的贡献 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">图</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">基线</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">回答的问题</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图一</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>CPU 单线程 float32</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使用 NPU 的整体收益</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图二</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1（float，基础 API 组合）</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每一项改动各自的贡献</td>
</tr>
</tbody>
</table>

读图二时请把两条线索分开看：v2 与 v1 的差别只是写法，v3、v4 与 v1 的差别是数据类型。前者的柱高变化反映高阶 API 的实现代价，后者反映搬运量减半在核函数一侧兑现了多少。两张图都以核函数耗时为口径；读图时须同时对照结果表中的 `cpu_ms` 一列，因为加速比的高低既取决于 NPU 侧，也取决于基准实现的快慢（实验二 §13 ⑤）。

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_KERNEL, C_ALT, C_BASE, C_OPT = "#3B6FE0", "#E07A3B", "#9AA5B1", "#2E9E6B"

vers = [r["ver"] for r in rows_main]
spk = [r["sp_kernel"] for r in rows_main]
xpos = np.arange(len(vers))

fig, ax = plt.subplots(figsize=(7.6, 4.3), dpi=120)
ax.bar(xpos, spk, 0.5, color=C_KERNEL, label="NPU kernel only")
ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
ax.text(-0.45, 1.06, "baseline: CPU 1 thread = 1.0x", fontsize=9, color="#777777")
for i, a in enumerate(spk):
    ax.text(i, a, "%.1fx" % a, ha="center", va="bottom", fontsize=8)
ax.set_xticks(xpos)
ax.set_xticklabels(
    ["%s\n(%s)" % (r["ver"], chk_main[r["ver"]]["dtype"]) for r in rows_main]
)
ax.set_yscale("log")
ax.set_ylabel("Speedup over CPU baseline")
ax.set_title("Lab 4: Sigmoid - NPU kernel speedup vs single-thread CPU")
ax.grid(axis="y", alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
sp_v1 = [base_v1 / r["kernel_ms"] for r in rows_main]
colors = [C_BASE, C_KERNEL, C_OPT, C_OPT]

fig, ax = plt.subplots(figsize=(7.0, 4.0), dpi=120)
ax.bar(vers, sp_v1, 0.55, color=colors[: len(vers)])
ax.axhline(2.0, color="#888888", lw=1.0, ls="--")  # 搬运量之比给出的上界
ax.text(
    len(vers) - 0.5,
    2.03,
    "transfer-size ratio 2.0x",
    fontsize=9,
    color="#777777",
    ha="right",
)
for i, v in enumerate(sp_v1):
    ax.text(i, v, "%.2fx" % v, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Speedup vs. v1 (float, base API)")
ax.set_title("Lab 4: contribution of each change")
ax.grid(axis="y", alpha=0.3)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 11. 误差分析：把两类失效分开

上表中 v3 与 v4 的最大相对误差都可能很大，但成因不同。本节按 §3.1 的两条阈值把误差分段，再用那条 1/2 判据把二者分开。

三段的含义见 §3.1 的表格。`over_half` 一列是「相对误差超过 1/2 的元素个数」，该列大于 0 即说明区间内的误差不可能只来自输出量化。

In [ ]:
def seg_table(rows, vers=("v1", "v3", "v4")):
    name = {1: "① 输出下溢", 2: "② 中间量溢出", 3: "③ 正常区间"}
    print(
        "%-5s %-14s %20s %10s %14s %12s"
        % ("版本", "区间", "x 范围", "元素数", "最大相对误差", "超过 1/2 的个数")
    )
    print("-" * 92)
    for ver in vers:
        for r in rows:
            if r["ver"] != ver:
                continue
            s = int(r["seg"])
            if r["count"] == 0:  # 取值范围较小时前两段可能为空，跳过
                continue
            print(
                "%-5s %-14s %9.2f .. %-8.2f %10d %14.3e %12d"
                % (
                    ver,
                    name[s],
                    r["x_lo"],
                    r["x_hi"],
                    int(r["count"]),
                    r["max_rel"],
                    int(r["over_half"]),
                )
            )
        print()


seg_table(seg_main)

再看误差随 x 的分布。下图的每个点是一个宽度为 0.25 的区间内的**最大**相对误差，两条竖线即 §3 的两条阈值。

三条曲线要分别读：`v1` 是 float 版本，作为参照；`v3` 与 `v4` 的输入输出都是 `half`，差别只在中间计算的精度。

In [ ]:
fig, ax = plt.subplots(figsize=(10.4, 4.8), dpi=120)

STYLE = [
    ("v1", "v1  float, base API", C_BASE, "-"),
    ("v3", "v3  half, base API", "#C7000B", "-"),
    ("v4", "v4  half io + float calc", C_OPT, "-"),
]
for ver, label, color, ls in STYLE:
    if ver not in bins_main:
        continue
    lo, hi, rel = bins_main[ver]
    edges = np.linspace(lo, hi, len(rel) + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    rel = np.array(rel)
    rel[rel <= 0] = 1e-16  # 对数坐标下把 0 挪到图外
    ax.semilogy(centers, rel, ls, lw=1.8, color=color, label=label)

ax.axhline(0.5, color="#888888", lw=1.2, ls="--")
ax.text(
    base_main["xrange"],
    0.75,
    "criterion: relative error = 1/2",
    fontsize=9,
    color="#666666",
    ha="right",
)
for xv, txt in ((base_main["x_ovf"], "x_ovf"), (base_main["x_unf"], "x_unf")):
    ax.axvline(xv, color="#7A3FA8", lw=1.1, ls=":")
    ax.text(
        xv,
        3e3,
        " %s = %.2f" % (txt, xv),
        fontsize=9,
        color="#7A3FA8",
        rotation=90,
        va="top",
    )

ax.set_ylim(1e-8, 1e5)
ax.set_xlabel("input x")
ax.set_ylabel("max relative error in bin")
ax.set_title("Lab 4: relative error vs. input range")
ax.grid(alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper right", fontsize=9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 12. 参数扫描：核数

固定分块长度，改变参与计算的核数，同时记录 v1（float）与 v3（half）的耗时。两个版本的计算逻辑逐字相同，差别只有数据类型，因此这组数据回答的是：**搬运量减半之后，核数的作用区间是否发生了变化。**

这组数据还有一个用途：观察 `v1/v3` 这个比值随核数如何变化。若它随核数增大向 1 收敛，说明核函数耗时中与数据类型无关的那一部分（启动与调度的固定开销）占比在上升。§13 ② 用的正是这组数据。

> **不要用两端的两个点去解「固定开销 + 可并行部分」这类两参数模型。** 形如 t(b) = F + S/b 的模型只需两个点即可解出，看上去很省事，但它把全部结论压在 b = 1 与最大核数这两个点上，而 b = 1 恰是整条曲线上噪声最大的一档——该档只启动一个核，单次运行的抖动、机器上的其它负载都会被完整放大。换一台机器就可能得到相反的结论。若确实需要拆分固定开销，应当对全部档位做最小二乘拟合，并先确认曲线本身没有明显的离群点。

扫描每档只运行一次，个别档位可能出现明显偏离趋势的点。判断趋势应看整体走向，不要在单个离群点上做解释。

In [ ]:
_built = {}


def build_and_run(defines=None, args=(), tag="scan"):
    # 与实验二、三同名同约定；同一个 tag 只编译一次
    exe = "src_sigmoid/ascendc_sigmoid_%s" % tag
    if tag not in _built:
        cmd = ["bisheng", SRC] + FLAGS + ["-o", exe]  # 与 §9 使用同一组编译选项
        for k, v in (defines or {}).items():
            cmd.append("-D%s=%s" % (k, v))
        b = subprocess.run(cmd, capture_output=True, text=True)
        _built[tag] = b.returncode == 0
        if not _built[tag]:
            print("❌ 编译失败（%s）：%s" % (tag, (b.stdout + b.stderr).strip()[-300:]))
    if not _built[tag]:
        return "", False
    r = subprocess.run(
        ["./" + exe] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=900,
    )
    # 程序在「有版本未通过校验」时返回 1，这不是运行失败。
    # 判断是否正常运行要看有没有产出 [PERF] 记录行，而不是看返回码
    return r.stdout, ("[PERF]" in r.stdout)


block_dims = [1, 2, 4, 8, 16, 32]
scan_core = []

for bd in block_dims:
    txt, ok = build_and_run({"LAB_BLOCK_DIM": bd}, tag="bd%d" % bd)
    if not ok:
        print("❌ blockDim=%-3d 编译或运行失败（可能超出设备可用核数）" % bd)
        continue
    p = {r["ver"]: r for r in parse_perf(txt)}
    scan_core.append((bd, p["v1"]["kernel_ms"], p["v3"]["kernel_ms"]))
    print(
        "blockDim=%-3d v1(float)=%.4f ms  v3(half)=%.4f ms  v1/v3=%.3fx"
        % (
            bd,
            p["v1"]["kernel_ms"],
            p["v3"]["kernel_ms"],
            p["v1"]["kernel_ms"] / p["v3"]["kernel_ms"],
        )
    )

In [ ]:
if scan_core:
    bd = [s[0] for s in scan_core]
    ms1 = [s[1] for s in scan_core]
    ms3 = [s[2] for s in scan_core]
    sp1 = [ms1[0] / m for m in ms1]
    sp3 = [ms3[0] / m for m in ms3]
    ratio = [a / b for a, b in zip(ms1, ms3)]

    fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.2), dpi=120)

    ax = axes[0]
    ax.plot(bd, sp1, marker="o", lw=2, color=C_KERNEL, label="v1 float")
    ax.plot(bd, sp3, marker="s", lw=2, color=C_ALT, label="v3 half")
    ax.plot(bd, bd, lw=1, ls="--", color="#bbbbbb", label="ideal linear")
    ax.set_xscale("log", base=2)
    ax.set_xticks(bd)
    ax.set_xticklabels(bd)
    ax.set_xlabel("blockDim (number of vector cores)")
    ax.set_ylabel("Speedup vs. blockDim = 1")
    ax.set_title("Lab 4: core-count scaling")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, loc="upper left")

    ax = axes[1]
    ax.plot(bd, ratio, marker="D", lw=2, color=C_OPT)
    ax.axhline(2.0, color="#888888", lw=1.0, ls="--")
    ax.text(
        bd[-1],
        2.04,
        "transfer-size ratio 2.0x",
        fontsize=9,
        color="#777777",
        ha="right",
    )
    for x, y in zip(bd, ratio):
        ax.text(x, y, "%.2fx" % y, ha="center", va="bottom", fontsize=8)
    ax.set_xscale("log", base=2)
    ax.set_xticks(bd)
    ax.set_xticklabels(bd)
    ax.set_xlabel("blockDim (number of vector cores)")
    ax.set_ylabel("v1 time / v3 time  ( >1 means half faster )")
    ax.set_title("Lab 4: benefit of half vs. core count")
    ax.grid(alpha=0.3)

    for a in axes:
        for s in ("top", "right"):
            a.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

## 13. 结果分析

> 以下结论针对**趋势规律**。具体数值随硬件规格、CANN 版本与系统负载而变化，请以本机的运行输出为准。

**① v1 → v2：高阶 API 以一部分耗时换取数量级的精度提升**

官方明确指出**高阶 API 通过调用基础 API 来实现功能**，因此高阶 `Sigmoid` 的内部同样是若干条基础指令的组合，耗时与手工组合处于同一量级。但两者并不等价，读数据时请同时看两列：耗时上 v2 通常慢一些，因为它的内部实现要保证数值质量；精度上 v2 的绝对误差应当比 v1 低若干个数量级，该差距远超测量噪声，其来源见 ⑥。

这一步的取舍因此是可以量化的：**以一部分耗时换取若干个数量级的精度**。§2.2 表格中「数值处理：手工组合由开发者负责 / 高阶 API 由实现负责」一行的分量即在于此。

**② v1 → v3：改用 half，核函数一侧几乎没有收益**

`half` 版本每元素读写 4 字节而非 8 字节。若耗时完全由字节数决定，加速比的上界就是 2 倍。**核函数口径下的实测通常远达不到这个上界**：在主对照的核数配置上，v3 相对 v1 往往只快百分之几；§12 的核数扫描进一步显示，这个比值随核数增大还会继续向 1 靠拢，在核数较大的档位上甚至可能反转。

原因有两层，都与搬运量无关：

- **最重的那一步计算量并没有减半。** §1 把 Sigmoid 拆成四条指令，但其中的 `Exp` 是超越函数，代价远高于一次浮点加法；把它计为「一次运算」会低估计算量，从而低估算术强度。更直接的依据是 §1 引用的那条官方说明：**`Exp`、`Ln` 接口处理同样数量的 `half` 与 `float` 数据，耗时是一样的**，其内部对 `float` 做了优化。也就是说这条链上最重的一步在两个版本中完全等价，改数据类型对它没有任何作用。
- **核数越多，固定开销的占比越高。** 核函数耗时中有一部分是启动与调度，与数据类型和数据量都无关。核数增大时可并行的那一部分被摊薄，固定开销的占比随之上升，两个版本的耗时因而向同一个数值收敛。

**该算子从字节数看像是访存密集型，实测却有相当比重的时间花在计算上。** 由此得到一条判据：判断一个算子是否访存受限，比估算算术强度更可靠的办法是**改变数据类型、观察耗时是否按比例变化**，本节做的正是这件事，而它给出的答案是否定的。

> **搬运量减半这项收益并没有消失，只是不落在核函数这一段。** 主机与设备之间的数据搬运时间与字节数成正比，传输量减半即意味着传输时间减半。但这段时间是否应当计入算子的收益，取决于数据是否真的需要往返主机——按实验二 §5.1 与实验三 §15 ④ 的口径，真实计算图中数据一次搬入后常驻设备内存，这一段并不逐算子发生。顺带可以理解本算子在这条通路上为何比前两个实验有利：Sigmoid 每个元素在 CPU 上要算一次 `expf`，CPU 侧的代价比向量加法与规约高一个数量级，而搬运量并没有相应增加——**计算越重的算子，搬运在总时间中的占比越低**。数据通路本身的性质属于应用开发的范畴，第七章会系统讨论。

**③ 绝对误差与相对误差可能给出相反的结论**

请对照 §9.1 的表格确认两件事：v3 的**最大绝对误差**是否落在容差之内、因而判为 `PASS`；以及它的**最大相对误差**是否等于 1（只要输入取值范围覆盖到 x < −16.64，该值必然为 1，原因见 ④ 中的区间 ①）。

若两者同时成立，则出现了一个「通过校验但在部分区间内结果完全错误」的版本。绝对误差之所以掩盖了问题，是因为出错的区间恰好是真值接近 0 的区间：真值只有 10^-8 量级时，即使结果为 0，绝对误差也只有 10^-8。**若只用绝对误差判定，这一缺陷不会被发现。**

由此得到一条更一般的结论：**校验口径决定了哪些缺陷会被发现。** 判定口径应当依据输出的**下游用途**选取，而不是依据哪个口径更容易通过——若该输出接下来要送入 `Log`，真值接近 0 的那一段就变得至关重要，此时必须改用相对误差。

**④ 两类数值失效必须分开：一类可修复，一类不可修复**

v4 的最大相对误差同样接近 1。若只看这一个数字，会误以为混合精度没有起作用。§11 的分段统计给出了正确的判断依据，读表时按三段分别看：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 区间 | 应当观察到的 | 含义 |
| --- | --- | --- |
| ③ x > −11.09 | v3 与 v4 的最大相对误差应当接近 | 中间量未溢出，两者受同样的因素限制（见 ⑥） |
| ② −16.64 < x < −11.09 | **`over_half` 一列：v3 应当等于该段的全部元素数，v4 应当等于 0** | v3 的 e^(−x) 溢出；v4 只受非规格化数量化影响 |
| ① x < −16.64 | 两者的最大相对误差都应当为 1 | 真值小于 `half` 最小非规格化数，属格式的固有极限 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">区间</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">应当观察到的</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">③ x &gt; −11.09</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 与 v4 的最大相对误差应当接近</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">中间量未溢出，两者受同样的因素限制（见 ⑥）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">② −16.64 &lt; x &lt; −11.09</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**<code>over_half</code> 一列：v3 应当等于该段的全部元素数，v4 应当等于 0**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 的 e^(−x) 溢出；v4 只受非规格化数量化影响</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">① x &lt; −16.64</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两者的最大相对误差都应当为 1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">真值小于 <code>half</code> 最小非规格化数，属格式的固有极限</td>
</tr>
</tbody>
</table>

区间 ② 是**唯一能够区分算法优劣的窗口**。需要注意的是，在该窗口内 v4 的相对误差也不会很小：`half` 非规格化数的步长恒为 5.96×10^-8，而真值本身只有 10^-7 ~ 10^-5 量级，量化误差的相对占比可达数十个百分点。**判别的依据不是误差的大小，而是误差是否超过了那条可证明的上界 1/2。**

混合精度消除的是区间 ② 的失效，区间 ① 无法消除。把三段混在一起统计，只会得到「两个版本都很差」这一无用的结论——**单一的全局误差指标不足以评价数值质量**。

**⑤ v3 → v4：混合精度的代价落在核函数一侧**

v4 相对 v3 多做了三件事：两次 `Cast`、在 float 域（而非 half 域）完成四步运算（矢量运算的数据量翻倍）、片上占用由 40 KB 升到 64 KB。前两件都是实际增加的计算量，因此**核函数口径下 v4 应当明显慢于 v3**。请核对这一点：若确实如此，说明在本实验的规模与核数下，该算子并不像算术强度估计的那样纯粹受访存制约，多出的计算量是可以被测出的——这与 ② 的结论互相印证。

需要一并记住的是这笔开销买到了什么：区间 ② 的失效被完全消除（见 ④）。**这是一笔可以量化的交换——以核函数一侧可测出的额外计算，换取一整段输入区间上的数值正确性。** 是否值得，取决于该区间在实际输入分布中所占的比例；若输入本来就不会取到 −11.09 以下，这笔开销就是纯粹的浪费。动手练习第 1 题即为该情形。

**⑥ 精度的瓶颈不在数据类型，而在链上的一条基础接口**

`[BASE]` 行给出了 CPU `float32` 基准相对 `double` 真值的误差，它是「用 float32 计算该算子所能达到的精度」的参照线。请把两个 `float` 版本与它对照：v2 应当与参照线同量级；若 v1 高出若干个数量级——同样的数据类型却差了这样大的差距——说明实现链上有一环拉低了整体精度。

定位这一环的办法，是找一个能让其余各环的误差被自然吸收的输入点。在 x = 20 处，e^(−x) = 2.1×10^-9，而 float32 在 1.0 附近的间隔为 1.2×10^-7，因此 `Adds` 算出的 1 + e^(−x) 在 float32 下**精确等于 1.0**——`Muls` 与 `Exp` 两步的误差在此被完全吸收，输出只经过 `Reciprocal(y, 1.0)` 一步。若 v1 在该处的相对误差与其余位置相同（§11 图中表现为一条与 x 无关的水平线），则误差只能来自 `Reciprocal`：它是硬件提供的**近似**求倒数接口，其相对精度固定，与被求倒数的值无关。

v3 与 v4 在区间 ③ 上同样会收敛到这条水平线，因为它们的链上有同一步，而 `half` 的输出分辨率比它更小、被完全覆盖——这正是 §4.2 把这三个版本的容差归入同一档的依据。由此得到本节的结论：手工组合基础 API 时，**链上每一条接口自身的实现精度都会计入最终结果**。动手练习第 4 题用 `Div` 替换 `Reciprocal`，用以检验这一判断。



---

### 🎓 结论

Sigmoid 揭示了算子开发的第四个核心问题：**数据类型与接口的选择**。其结论可以概括为四条——

**其一**，高阶 API 与数据类型是两个独立的维度：前者决定代码怎么写、数值处理由谁负责，后者决定搬运量与可表示的范围；

**其二**，中间量的取值范围与输入输出的取值范围是两个独立的问题，必须分别核算；且必须把可修复的算法缺陷与格式的固有极限区分开，办法是分段统计加一条可证明的上界；

**其三**，手工组合基础 API 时，链上每一条接口自身的实现精度都会计入结果。容差应当由这条链上最弱的一环推出，而不是由数据类型的机器精度推出；

**其四**，低精度数据类型的收益不会自动兑现——它只作用于耗时中与字节数成正比的那一部分。本算子最重的一步是超越函数，其代价与数据类型无关，核函数一侧因而几乎没有收益。报告任何一项优化的收益时，都必须说明它究竟落在哪一段上。

## 14. 🔧 动手练习

> **提示**：修改源码需要从 §7.1 开始按顺序重新执行全部写入单元格。只改参数的练习用 `build_and_run({...}, tag='...')` 一行即可完成；只改规模或取值范围的练习连重新编译都不需要，`run_demo([n, xrange])` 即可。

1. **缩小输入取值范围**。运行 `run_demo([2097152, 8])`，即把输入限制在 [−8, 8]。此时 v3 的最大相对误差是多少？请解释为何变化如此之大。*提示：先看 §11 分段表里三段的元素数各是多少——取值范围小于 11.09 时，前两段是空的，`seg_table` 会直接跳过它们。*

2. **就地复用中间缓冲**。把 `KernelSigmoidBaseApi` 的四步变换全部改为就地形式，直接在 `yLocal` 上完成，去掉 `tmpBuf`。结果是否仍然正确？片上占用下降了多少？再对 v2 的高阶 `Sigmoid` 做同样的改动（把 `yLocal` 同时当作源和目的），记录二者的差异，并说明基础 API 与高阶 API 在地址重叠上的规定为何不同。

3. **把片上缓冲用尽**。用 `build_and_run({'LAB_TILE_LENGTH': tl}, tag='tl%d' % tl)` 把分块长度逐步增大到 8192、16384，观察 v2 从哪一档开始报错或给出错误结果。请先按 §6 的表核算每一档的片上占用，再据此说明「为高阶 API 预留临时空间」这句话的具体含义。*注意 v1 的占用是 v2 的 1.25 倍，可能先于 v2 失败。*

4. **换一种求倒数的写法（§13 ⑥ 的决定性检验）**。把 `KernelSigmoidBaseApi` 与 `KernelSigmoidMixed` 中的 `Reciprocal(y, t)` 换成 `Div(y, ones, t, ...)`（需要额外一块常量为 1 的缓冲，可用 `Duplicate` 填充）。重新运行后回答三件事：v1 的绝对误差降到了什么量级？§11 图中那条水平线是否消失？§4.2 第二档的容差可以收到多少？再比较改动前后的耗时与片上占用，说明这笔精度是用什么换来的。

5. **核数扫描的对照**。§12 已经给出 v1 与 v3 的曲线。请补上 v4，并回答：v4 的曲线更接近 v1 还是 v3？据此判断 v4 多出来的两次 `Cast` 究竟有没有改变它的瓶颈类型。

6. **量化两条阈值**。写一小段 Python，对 x 从 −20 到 0 逐点计算 `np.float16` 下的 `1/(1+np.exp(-x, dtype=np.float16))`，与 `np.float64` 的结果比较，找出相对误差首次超过 1/2 的位置。它与 §3 推出的 −11.09 是否一致？若不一致，差在哪里？

7. **【进阶】数值稳定的分段形式**。用 Ascend C 的比较与选择类接口实现分段形式：x ≥ 0 时用 1/(1+e^(−x))，x < 0 时用 e^x/(1+e^x)。在纯 `half` 域内验证：这种写法能否消除区间 ② 的失效？区间 ① 呢？请把结论与 v4 的做法对照。

8. **【进阶】换成 bfloat16**。若把输出数据类型由 `half` 改为 `bfloat16`（8 位指数、7 位尾数），中间量溢出与输出下溢这两类失效各会如何变化？请先按 §3 的方法推算两条阈值，再说明本实验的哪些结论需要改写。

## 15. 🤔 思考题

1. 为什么中间量 e^(−x) 的取值范围可以远大于输入与输出的取值范围？在设计一个算子时，应当在哪一步核算中间量的范围？

2. v3 通过了绝对误差校验，却在部分区间内给出了几乎完全错误的结果。如果不做 §11 的分段统计，还有哪些手段可以发现这类缺陷？

3. 高阶 API 的「接口框架申请临时空间」要求预留未分配的片上空间。若一个算子已把片上缓冲用尽，应当如何改造？改造之后又引入了什么新的负担？

4. v4 的两块 `float` 中间缓冲是否可以合并为一块？若可以，需要满足什么条件？*（提示：查阅所用接口关于源、目的地址重叠的约束）*

5. 把 `Exp` 计为一次运算时，本算子的算术强度约为 0.5 Op/Byte，据此会判定它访存受限；但 §13 ② 的实测并不支持这个判定。请说明这个估计错在哪里，以及应当如何修正对 `Exp` 的计数。若把 Sigmoid 换成一个需要二十条基础 API 的复杂函数，v3 相对 v1 的加速比会更高还是更低？

6. 实验三把容差由 1e-5 放宽到 1e-4，理由是规约改变了累加顺序、误差随规模累积；本实验把含 `Reciprocal` 的三个版本的容差放宽到 5e-3，理由却完全不同。请说明这两处放宽在性质上的区别，以及各自应当如何随参数变化。

7. §3.1 给出「区间 ② 内纯量化所致的相对误差不超过 1/2」这一上界。请完成该上界的推导，说明为何它对硬件在溢出时饱和还是产出 inf 都成立，并据此解释：v3 与 v4 的最大相对误差都接近 1，为何只有 v3 存在算法缺陷。

8. 假设某算子的输出将被送入 `Log` 运算。此时应当采用绝对误差还是相对误差作为判定口径？请给出理由，并说明这会如何改变本实验对 v3 的评价。

9. Host 侧的 `F32ToF16` 在尾数进位时写的是 `++halfMan`，没有单独处理「进位溢出到指数位」的情形。请说明为什么这样写是正确的。*（提示：把 halfMan 与指数位拼接的那一步写成一次加法）*

10. 本实验用 `F16ToF32(F32ToF16(x))` 作为 `half` 版本的输入真值来源。若直接用 `float` 的输入算真值，误差会被高估多少？该高估量与算子本身的误差是否可分？

## 16. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 复合函数的实现 | 一行数学公式对应多条基础矢量指令，并需要中间缓冲 |
| 中间缓冲的判据 | 不跨流水任务，故用 `VECCALC` 位置的 `TBuf`，与实验三一致 |
| 中间量的量程 | 与输入输出的量程是两个独立的问题，必须分别核算 |
| 官方的 API 分层 | 基础 API 抽象单指令，高阶 API 抽象**单核公共算法**并通过调用基础 API 实现；`Sigmoid` 属于高阶 API 中的**激活函数**一类 |
| 高阶 API | 把常见算法收敛为一次调用，保证跨硬件版本兼容性（基础 API 中标注 ISASI 的部分不保证） |
| 临时空间的两种取得方式 | 开发者传入可复用、利用率高；框架申请无须估算，但**必须预留空间** |
| 地址重叠约束 | 基础算术 API 未禁止源与目的重叠，高阶 API 明确禁止；就地复用前须查文档 |
| `half` 的收益 | 每元素搬运量减半，加速比上界为 2 倍；实际兑现多少取决于耗时中与字节数成正比的比例 |
| 是否访存受限的判据 | 比估算算术强度更可靠的办法：改变数据类型，观察耗时是否按比例变化 |
| `half` 的代价一 | 输出的舍入误差上界约 4.9×10^-4（半个 ulp，1 ulp = 9.8×10^-4），另有 5.96×10^-8 的下界（x < −16.64 时输出必为 0）。均为格式的固有极限，**不可修复** |
| `half` 的代价二 | 中间量溢出：上限 65504，对应 x < −11.09。**可通过混合精度修复** |
| `Cast` 与混合精度 | 输入输出低精度保吞吐，中间计算高精度保正确性 |
| 两套真值 | 量化后的输入要配量化后的真值，否则会把输入量化误差记到算子头上 |
| 判定口径与分析口径 | 二者可以不同：绝对误差用于判定，相对误差分段用于分析 |
| 容差的来源 | 由**链上精度最低的那一条接口**决定，而不是由数据类型的机器精度决定 |
| 基础接口的实现精度 | `Reciprocal` 一类近似接口会把整条链的精度拉到它的水平；高阶 API 由实现负责数值处理 |
| 定位精度瓶颈的办法 | 找一个能让其余各环误差被自然吸收的输入点，剩下的那一环即暴露 |
| 低精度收益的兑现条件 | 只作用于耗时中与字节数成正比的那一部分。本算子最重的一步是超越函数，官方明确其对 `half` 与 `float` 等时，核函数一侧因而几乎没有收益 |
| 区分两类失效的判据 | 区间 ② 内相对误差是否**超过 1/2**：量化误差有此可证明上界，计算出错则没有 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">复合函数的实现</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一行数学公式对应多条基础矢量指令，并需要中间缓冲</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">中间缓冲的判据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不跨流水任务，故用 <code>VECCALC</code> 位置的 <code>TBuf</code>，与实验三一致</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">中间量的量程</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与输入输出的量程是两个独立的问题，必须分别核算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方的 API 分层</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础 API 抽象单指令，高阶 API 抽象<strong>单核公共算法</strong>并通过调用基础 API 实现；<code>Sigmoid</code> 属于高阶 API 中的<strong>激活函数</strong>一类</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把常见算法收敛为一次调用，保证跨硬件版本兼容性（基础 API 中标注 ISASI 的部分不保证）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">临时空间的两种取得方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">开发者传入可复用、利用率高；框架申请无须估算，但<strong>必须预留空间</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">地址重叠约束</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础算术 API 未禁止源与目的重叠，高阶 API 明确禁止；就地复用前须查文档</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>half</code> 的收益</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每元素搬运量减半，加速比上界为 2 倍；实际兑现多少取决于耗时中与字节数成正比的比例</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是否访存受限的判据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">比估算算术强度更可靠的办法：改变数据类型，观察耗时是否按比例变化</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>half</code> 的代价一</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出的舍入误差上界约 4.9×10^-4（半个 ulp，1 ulp = 9.8×10^-4），另有 5.96×10^-8 的下界（x &lt; −16.64 时输出必为 0）。均为格式的固有极限，<strong>不可修复</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>half</code> 的代价二</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">中间量溢出：上限 65504，对应 x &lt; −11.09。<strong>可通过混合精度修复</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Cast</code> 与混合精度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入输出低精度保吞吐，中间计算高精度保正确性</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两套真值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">量化后的输入要配量化后的真值，否则会把输入量化误差记到算子头上</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">判定口径与分析口径</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">二者可以不同：绝对误差用于判定，相对误差分段用于分析</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">容差的来源</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由<strong>链上精度最低的那一条接口</strong>决定，而不是由数据类型的机器精度决定</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础接口的实现精度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Reciprocal</code> 一类近似接口会把整条链的精度拉到它的水平；高阶 API 由实现负责数值处理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">定位精度瓶颈的办法</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">找一个能让其余各环误差被自然吸收的输入点，剩下的那一环即暴露</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">低精度收益的兑现条件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只作用于耗时中与字节数成正比的那一部分。本算子最重的一步是超越函数，官方明确其对 <code>half</code> 与 <code>float</code> 等时，核函数一侧因而几乎没有收益</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">区分两类失效的判据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">区间 ② 内相对误差是否<strong>超过 1/2</strong>：量化误差有此可证明上界，计算出错则没有</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **数据类型的选择不是一个局部决定，它同时影响搬运量、片上占用与数值正确性三件事。**

本实验中，`float` 改 `half` 这一处改动带来了四项后果：搬运量减半、片上占用减半、输出分辨率下降、以及中间量溢出。**只关注其中任何一项，都会得出片面的结论。**

§13 ⑥ 给出了一条并列的原则：

> **接口的选择同样不是一个局部决定——手工组合基础 API 时，链上每一条接口自身的实现精度都会计入最终结果。**

本实验中，正是一条近似求倒数接口，使 `float` 版本的精度低于 `half` 的输出分辨率。

### 与后续实验的衔接

➡️ **后续内容：实验五 · 融合算子**。前四个实验各自实现了一个独立的算子。但在实际的神经网络中，算子往往一个接一个执行，每一次都要把数据从 Global Memory 读进来、算完再写回去。当算子处于访存受限区域时，这些往返本身就是最大的开销。下一个实验将把多个算子合并为一个，从**减少访存量**这一方向继续优化——这正是实验二 §13 与实验三 §15 ⑦ 都指向的那条结论。